In [ ]:
# Auto-alias Kaggle datasets to flat paths under /kaggle/working/inputs/.
# /kaggle/input/ is read-only and Kaggle's newer mount uses owner-prefixed paths
# (/kaggle/input/datasets/<owner>/<slug>/) plus a separate /kaggle/input/competitions/
# for competition data. We symlink everything to flat /kaggle/working/inputs/<slug>/
# so the rest of the notebook can use clean uniform paths.
import os, glob, shutil

SYMLINK_DIR = '/kaggle/working/inputs'
os.makedirs(SYMLINK_DIR, exist_ok=True)

# Collect real paths: owner-prefixed datasets + competition data
SLUG_TO_REAL = {}
for owner_dir in sorted(glob.glob('/kaggle/input/datasets/*')):
    if os.path.isdir(owner_dir):
        for ds in sorted(glob.glob(os.path.join(owner_dir, '*'))):
            if os.path.isdir(ds):
                SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/competitions/*')):
    if os.path.isdir(ds):
        SLUG_TO_REAL[os.path.basename(ds)] = ds
# Also fold in any flat-mounted datasets (forward-compat)
for ds in sorted(glob.glob('/kaggle/input/*')):
    name = os.path.basename(ds)
    if name in ('datasets', 'competitions'):
        continue
    if os.path.isdir(ds) and name not in SLUG_TO_REAL:
        SLUG_TO_REAL[name] = ds

# Create symlinks (overwrite if already there from a previous run)
for slug, real in SLUG_TO_REAL.items():
    flat = os.path.join(SYMLINK_DIR, slug)
    if os.path.lexists(flat):
        try:
            if os.path.islink(flat):
                os.unlink(flat)
            elif os.path.isdir(flat):
                shutil.rmtree(flat)
            else:
                os.unlink(flat)
        except OSError as e:
            print(f'WARN: could not clean existing {flat}: {e}')
    os.symlink(real, flat)

print(f'Created {len(SLUG_TO_REAL)} symlinks under {SYMLINK_DIR}:')
for slug in sorted(SLUG_TO_REAL):
    print(f'  {SYMLINK_DIR}/{slug} -> {SLUG_TO_REAL[slug]}')
    



In [ ]:
!python --version

!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/addict-2.4.0-py3-none-any.whl
# !pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/mmengine-0.7.4-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mmengine-0.8.3-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/mmcv-2.0.0-cp310-cp310-linux_x86_64.whl
!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/terminaltables-3.1.10-py2.py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/pycocotools-206/wheels/pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mmdet-3.1.0-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/ensemble_boxes-1.0.9-py3-none-any.whl

In [ ]:
!ls /kaggle/input/ | sort
!echo "---"
!find /kaggle/input -name "mmdet-*.whl" 2>/dev/null
!find /kaggle/input -name "mmdet*-3.1*.whl" 2>/dev/null


In [ ]:
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/ordered_set-4.1.0-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/model_index-0.1.11-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/einops-0.6.1-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mat4py-0.5.0-py2.py3-none-any.whl
!pip install --no-deps --no-index /kaggle/working/inputs/vasculature-packages/mmpretrain-1.0.1-py2.py3-none-any.whl

In [ ]:
import glob
import os

import mmengine


def prepare_dataset():
    coco = {
        'info': {},
        'categories': [{
            'id': 0,
            'name': 'blood_vessel',
        },{
            'id': 1,
            'name': 'glomerulus',
        },{
            'id': 2,
            'name': 'unsure'
        }],
        'annotations': []
    }
    test_imgs = glob.glob('/kaggle/working/inputs/hubmap-hacking-the-human-vasculature/test/*.tif')
    img_infos = []
    img_id = 0
    for path in test_imgs:
        filename = os.path.basename(path)
        img_info = dict(
            id=img_id,
            width=512,
            height=512,
            file_name=filename,
        )
        img_infos.append(img_info)
        img_id += 1
    coco['images'] = img_infos
    return coco


mmengine.dump(prepare_dataset(), '/kaggle/working/test.json')

In [ ]:
%%writefile test.py

# Copyright (c) OpenMMLab. All rights reserved.
import argparse
import os
import os.path as osp
import warnings
from copy import deepcopy

from mmengine import ConfigDict
from mmengine.config import Config, DictAction
from mmengine.runner import Runner

from mmdet.engine.hooks.utils import trigger_visualization_hook
from mmdet.evaluation import DumpDetResults
from mmdet.registry import RUNNERS
from mmdet.utils import setup_cache_size_limit_of_dynamo


# TODO: support fuse_conv_bn and format_only
def parse_args():
    parser = argparse.ArgumentParser(
        description='MMDet test (and eval) a model')
    parser.add_argument('config', help='test config file path')
    parser.add_argument('checkpoint', help='checkpoint file')
    parser.add_argument(
        '--work-dir',
        help='the directory to save the file containing evaluation metrics')
    parser.add_argument(
        '--out',
        type=str,
        help='dump predictions to a pickle file for offline evaluation')
    parser.add_argument(
        '--show', action='store_true', help='show prediction results')
    parser.add_argument(
        '--show-dir',
        help='directory where painted images will be saved. '
        'If specified, it will be automatically saved '
        'to the work_dir/timestamp/show_dir')
    parser.add_argument(
        '--wait-time', type=float, default=2, help='the interval of show (s)')
    parser.add_argument(
        '--cfg-options',
        nargs='+',
        action=DictAction,
        help='override some settings in the used config, the key-value pair '
        'in xxx=yyy format will be merged into config file. If the value to '
        'be overwritten is a list, it should be like key="[a,b]" or key=a,b '
        'It also allows nested list/tuple values, e.g. key="[(a,b),(c,d)]" '
        'Note that the quotation marks are necessary and that no white space '
        'is allowed.')
    parser.add_argument(
        '--launcher',
        choices=['none', 'pytorch', 'slurm', 'mpi'],
        default='none',
        help='job launcher')
    parser.add_argument('--tta', action='store_true')
    # When using PyTorch version >= 2.0.0, the `torch.distributed.launch`
    # will pass the `--local-rank` parameter to `tools/train.py` instead
    # of `--local_rank`.
    parser.add_argument('--local_rank', '--local-rank', type=int, default=0)
    args = parser.parse_args()
    if 'LOCAL_RANK' not in os.environ:
        os.environ['LOCAL_RANK'] = str(args.local_rank)
    return args


def main():
    args = parse_args()

    # Reduce the number of repeated compilations and improve
    # testing speed.
    setup_cache_size_limit_of_dynamo()

    # load config
    cfg = Config.fromfile(args.config)
    cfg.launcher = args.launcher
    if args.cfg_options is not None:
        cfg.merge_from_dict(args.cfg_options)

    # work_dir is determined in this priority: CLI > segment in file > filename
    if args.work_dir is not None:
        # update configs according to CLI args if args.work_dir is not None
        cfg.work_dir = args.work_dir
    elif cfg.get('work_dir', None) is None:
        # use config filename as default work_dir if cfg.work_dir is None
        cfg.work_dir = osp.join('./work_dirs',
                                osp.splitext(osp.basename(args.config))[0])

    if args.checkpoint != 'none':
        cfg.load_from = args.checkpoint

    if args.show or args.show_dir:
        cfg = trigger_visualization_hook(cfg, args)

    if args.tta:

        if 'tta_model' not in cfg:
            warnings.warn('Cannot find ``tta_model`` in config, '
                          'we will set it as default.')
            cfg.tta_model = dict(
                type='DetTTAModel',
                tta_cfg=dict(
                    nms=dict(type='nms', iou_threshold=0.5), max_per_img=100))
        if 'tta_pipeline' not in cfg:
            warnings.warn('Cannot find ``tta_pipeline`` in config, '
                          'we will set it as default.')
            test_data_cfg = cfg.test_dataloader.dataset
            while 'dataset' in test_data_cfg:
                test_data_cfg = test_data_cfg['dataset']
            cfg.tta_pipeline = deepcopy(test_data_cfg.pipeline)
            flip_tta = dict(
                type='TestTimeAug',
                transforms=[
                    [
                        dict(type='RandomFlip', prob=1.),
                        dict(type='RandomFlip', prob=0.)
                    ],
                    [
                        dict(
                            type='PackDetInputs',
                            meta_keys=('img_id', 'img_path', 'ori_shape',
                                       'img_shape', 'scale_factor', 'flip',
                                       'flip_direction'))
                    ],
                ])
            cfg.tta_pipeline[-1] = flip_tta
        cfg.model = ConfigDict(**cfg.tta_model, module=cfg.model)
        cfg.test_dataloader.dataset.pipeline = cfg.tta_pipeline

    # build the runner from config
    if 'runner_type' not in cfg:
        # build the default runner
        runner = Runner.from_cfg(cfg)
    else:
        # build customized runner from the registry
        # if 'runner_type' is set in the cfg
        runner = RUNNERS.build(cfg)

    # add `DumpResults` dummy metric
    if args.out is not None:
        assert args.out.endswith(('.pkl', '.pickle')), \
            'The dump file must be a pkl file.'
        runner.test_evaluator.metrics.append(
            DumpDetResults(out_file_path=args.out))

    # start testing
    runner.test()


if __name__ == '__main__':
    main()


In [ ]:
%%writefile tta_test.py
"""Custom horizontal-flip TTA inference for mmdet instance segmentation
models (mmdet's native DetTTAModel doesn't support seg masks).

For each test image: run inference on original + h-flipped, flip flipped
predictions back to original coords, then per-class NMS to dedupe.

Outputs plain-dict format compatible with the WBF cell."""
import argparse
import glob
import os

import mmcv
import mmengine
import numpy as np
import torch
from torchvision.ops import nms
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

parser = argparse.ArgumentParser()
parser.add_argument('cfg')
parser.add_argument('ckpt')
parser.add_argument('--out', required=True)
parser.add_argument('--nms-iou', type=float, default=0.5,
                    help='IoU threshold for per-class NMS merging orig+flip')
args = parser.parse_args()

register_all_modules()
model = init_detector(args.cfg, args.ckpt, device='cuda')

TEST_IMGS = sorted(glob.glob(
    '/kaggle/working/inputs/hubmap-hacking-the-human-vasculature/test/*.tif'))

test_meta = mmengine.load('/kaggle/working/test.json')
id_by_name = {os.path.basename(im['file_name']): im['id']
              for im in test_meta['images']}

results = []
for path in TEST_IMGS:
    img = mmcv.imread(path)
    H, W = img.shape[:2]

    with torch.no_grad():
        r_orig = inference_detector(model, img)
        img_flip = np.ascontiguousarray(img[:, ::-1, :])
        r_flip = inference_detector(model, img_flip)

    inst_o = r_orig.pred_instances
    inst_f = r_flip.pred_instances

    # Flip the flipped boxes back: x1', x2' = W - x2, W - x1
    bb_f = inst_f.bboxes.clone()
    bb_f[:, [0, 2]] = W - bb_f[:, [2, 0]]

    bboxes = torch.cat([inst_o.bboxes, bb_f])
    scores = torch.cat([inst_o.scores, inst_f.scores])
    labels = torch.cat([inst_o.labels, inst_f.labels])

    # Per-class NMS
    keep_idx = []
    for cls in labels.unique():
        cls_mask = (labels == cls).nonzero(as_tuple=True)[0]
        if len(cls_mask) == 0:
            continue
        nms_keep = nms(bboxes[cls_mask], scores[cls_mask], args.nms_iou)
        keep_idx.append(cls_mask[nms_keep])
    keep = torch.cat(keep_idx) if keep_idx else torch.zeros(0, dtype=torch.long)

    results.append(dict(
        img_id=id_by_name[os.path.basename(path)],
        img_path=path,
        ori_shape=(H, W),
        pred_instances=dict(
            bboxes=bboxes[keep].cpu(),
            scores=scores[keep].cpu(),
            labels=labels[keep].cpu(),
        ),
    ))

mmengine.dump(results, args.out)
print(f'TTA done: {len(results)} images, out={args.out}')

In [ ]:
!cp -r /kaggle/working/inputs/hubmap-2023-modules /kaggle/working/hubmap_modules

In [ ]:
import time; PIPELINE_T0 = time.time()

import os, sys, subprocess, traceback

LOG_PATH = '/kaggle/working/progress.log'
ERR_PATH = '/kaggle/working/error.log'

def log_stage(name, info=''):
    """Append a timestamped progress line with RAM/GPU/disk stats."""
    try:
        import psutil
        ram = psutil.virtual_memory()
        ram_str = f'ram={ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB'
    except Exception:
        ram_str = 'ram=?'
    try:
        import torch
        if torch.cuda.is_available():
            parts = []
            for i in range(torch.cuda.device_count()):
                alloc = torch.cuda.memory_allocated(i) / 1024**3
                reserv = torch.cuda.memory_reserved(i) / 1024**3
                parts.append(f'gpu{i}={alloc:.1f}/{reserv:.1f}GB')
            gpu_str = ' '.join(parts)
        else:
            gpu_str = 'no cuda'
    except Exception:
        gpu_str = 'gpu=?'
    try:
        s = os.statvfs('/kaggle/working')
        disk_str = f'disk_free={s.f_bavail * s.f_frsize / 1024**3:.1f}GB'
    except Exception:
        disk_str = 'disk=?'
    line = f'[{time.strftime("%H:%M:%S")} +{(time.time()-PIPELINE_T0)/60:.1f}m] {name} | {ram_str} | {gpu_str} | {disk_str}'
    if info:
        line += f' | {info}'
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line, flush=True)


def _excepthook(exc_type, exc_value, exc_tb):
    with open(ERR_PATH, 'a') as f:
        f.write(f'\n=== UNCAUGHT EXCEPTION at {time.strftime("%H:%M:%S")} ===\n')
        traceback.print_exception(exc_type, exc_value, exc_tb, file=f)
    sys.__excepthook__(exc_type, exc_value, exc_tb)
sys.excepthook = _excepthook


def _run_2gpu(args0, args1, role='mmdet'):
    log_stage(f'START {role}', f'args0={args0[-2:]} args1={args1[-2:]}')
    err0_path = f'/kaggle/working/stderr_{role}_g0.log'
    err1_path = f'/kaggle/working/stderr_{role}_g1.log'
    env0 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
    env1 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1'}
    with open(err0_path, 'wb') as e0, open(err1_path, 'wb') as e1:
        p0 = subprocess.Popen(args0, env=env0, stderr=e0)
        p1 = subprocess.Popen(args1, env=env1, stderr=e1)
        p0.wait(); p1.wait()
    log_stage(f'END {role}', f'rc0={p0.returncode} rc1={p1.returncode}')
    if p0.returncode or p1.returncode:
        with open(ERR_PATH, 'a') as out:
            for path in [err0_path, err1_path]:
                if os.path.exists(path):
                    out.write(f'\n=== {path} ===\n')
                    with open(path) as f:
                        out.write(f.read())
        raise RuntimeError(f'{role} subprocess failed: rc0={p0.returncode} rc1={p1.returncode}')


def test_pair(cfg, ck0, ck1, out0, out1):
    role = os.path.basename(out0).replace('.pkl', '') + '+' + os.path.basename(out1).replace('.pkl', '')
    _run_2gpu(
        ['python', 'tta_test.py', cfg, ck0, '--out', out0],
        ['python', 'tta_test.py', cfg, ck1, '--out', out1],
        role=role,
    )


# Clear previous run's logs (if any)
for p in [LOG_PATH, ERR_PATH]:
    if os.path.exists(p):
        os.remove(p)

log_stage('pipeline_start')

test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/m0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/m0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/m1i.pth',
    '/kaggle/working/m0i.pkl',
    '/kaggle/working/m1i.pkl',
)


In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/y0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/y0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/y1i.pth',
    '/kaggle/working/y0i.pkl',
    '/kaggle/working/y1i.pkl',
)


In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/r0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/r0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/r1i.pth',
    '/kaggle/working/r0i.pkl',
    '/kaggle/working/r1i.pkl',
)


In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/s0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/s0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/s1i.pth',
    '/kaggle/working/s0i.pkl',
    '/kaggle/working/s1i.pkl',
)


In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/sb0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/sb0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/sb1i.pth',
    '/kaggle/working/sb0i.pkl',
    '/kaggle/working/sb1i.pkl',
)


---
## ⭐ NEW SECTION: self-trained YOLOv8x-seg 4-fold ensemble stream

The **3 code cells below** (install ultralytics, write `predict_yolo.py`, run it) are **NEW** — they are not in the original 1st place release notebook. **Copy these into your Kaggle notebook between the last mmdet inference cell (`sb0i/sb1i`) and the WBF ensemble cell.**

Required Kaggle datasets (add to your notebook):
- `yolo-vasculature-folds` (your `fold0.pt`..`fold3.pt`)
- `ultralytics-offline-wheels` (offline ultralytics wheel(s))
---

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# >>> NEW CELL (added by you, not in original 1st place notebook)     <<<
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# Install ultralytics offline for self-trained YOLOv8x-seg 4-fold ensemble stream.
# Assumes a Kaggle dataset 'ultralytics-offline-wheels' exists with the ultralytics wheel(s).
# --no-deps to avoid breaking the mmdet 3.1 / torch / numpy stack installed in cells 0-1.
!pip install -q --no-deps --no-index /kaggle/working/inputs/ultralytics-wheels/*.whl
# <<< end of new cell <<<

In [ ]:
%%writefile predict_yolo.py
"""Run self-trained YOLOv8x-seg 4-fold inference and dump mmdet-compatible .pkl
files (yolo0.pkl..yolo3.pkl) into /kaggle/working/.

Accepts --folds 0,1 (etc) so the parent notebook can run two instances in
parallel on two GPUs."""
import argparse, glob, os
import torch
import mmengine
from ultralytics import YOLO

parser = argparse.ArgumentParser()
parser.add_argument('--folds', type=str, default='0,1,2,3',
                    help='comma-separated fold indices to run')
args = parser.parse_args()
folds_to_run = [int(x) for x in args.folds.split(',')]

YOLO_CKPTS = [f'/kaggle/working/inputs/hubmap-yolo26x-4fold/fold{i}.pt' for i in range(4)]
TEST_IMGS = sorted(glob.glob('/kaggle/working/inputs/hubmap-hacking-the-human-vasculature/test/*.tif'))
IMGSZ = 768

test_meta = mmengine.load('/kaggle/working/test.json')
id_by_name = {os.path.basename(im['file_name']): im['id'] for im in test_meta['images']}

for fold_idx in folds_to_run:
    ckpt = YOLO_CKPTS[fold_idx]
    model = YOLO(ckpt)
    fold_results = []
    for path in TEST_IMGS:
        r = model.predict(path, imgsz=IMGSZ, conf=0.001, iou=0.7,
                          retina_masks=True, half=True, verbose=False)[0]
        boxes_xyxy = r.boxes.xyxy.detach().cpu().float()
        scores = r.boxes.conf.detach().cpu().float()
        labels = torch.zeros_like(scores, dtype=torch.long)
        assert boxes_xyxy.numel() == 0 or boxes_xyxy.max() <= 512 + 1e-3, \
            f"YOLO box exceeds 512 px on {path}: max={boxes_xyxy.max()}"
        fold_results.append(dict(
            img_id=id_by_name[os.path.basename(path)],
            img_path=path,
            ori_shape=(512, 512),
            pred_instances=dict(bboxes=boxes_xyxy, scores=scores, labels=labels),
        ))
    mmengine.dump(fold_results, f'/kaggle/working/yolo{fold_idx}.pkl')
    del model
    torch.cuda.empty_cache()
print(f'YOLO inference done for folds: {folds_to_run}')


In [ ]:
# Run YOLO folds 0,1 on GPU 0 and folds 2,3 on GPU 1 in parallel.
_run_2gpu(
    ['python', 'predict_yolo.py', '--folds', '0,1'],
    ['python', 'predict_yolo.py', '--folds', '2,3'],
    role='yolo',
)


---
## ⭐ END OF NEW SECTION

The WBF ensemble cell **below** is also **MODIFIED** — the `results` list and `weights` list have 4 new entries (`yolo0..yolo3` with weight 2 each). All other cells (mask repredict, RLE encode, submission) are **unchanged** from the original 1st place notebook.
---

In [ ]:
log_stage('START wbf')
import torch
import mmengine
from ensemble_boxes import weighted_boxes_fusion

# ABLATION_MODE: change this for each submission
#   'baseline'   = 10 mmdet + 4 YOLO (should match V6 0.563)
#   'mmdet_only' = 10 mmdet only (YOLO excluded from WBF)
#   'yolo_only'  = 4 YOLO only
ABLATION_MODE = 'mmdet_only'

MMDET_NAMES   = ['r0i','r1i','s0i','s1i','m0i','m1i','y0i','y1i','sb0i','sb1i']
MMDET_WEIGHTS = [2,2,2,2,1,1,1,1,2,2]
YOLO_NAMES    = ['yolo0','yolo1','yolo2','yolo3']
YOLO_WEIGHTS  = [2,2,2,2]

if ABLATION_MODE == 'baseline':
    names, weights = MMDET_NAMES + YOLO_NAMES, MMDET_WEIGHTS + YOLO_WEIGHTS
elif ABLATION_MODE == 'mmdet_only':
    names, weights = MMDET_NAMES, MMDET_WEIGHTS
elif ABLATION_MODE == 'yolo_only':
    names, weights = YOLO_NAMES, YOLO_WEIGHTS
else:
    raise ValueError(f'unknown ABLATION_MODE: {ABLATION_MODE}')

print(f'WBF mode={ABLATION_MODE}, streams={len(names)}')

results = [mmengine.load(f'/kaggle/working/{n}.pkl') for n in names]

SCALER  = 10000
IOU_THR = 0.7  # V6 original, reverted from 0.5

for rs in zip(*results):
    boxes_list  = [(r['pred_instances']['bboxes'] / SCALER).tolist() for r in rs]
    scores_list = [r['pred_instances']['scores'].tolist()           for r in rs]
    labels_list = [r['pred_instances']['labels'].tolist()           for r in rs]
    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights, iou_thr=IOU_THR, conf_type='avg',
    )
    rs[0]['pred_instances'] = dict(
        bboxes=torch.from_numpy(boxes).float() * SCALER,
        scores=torch.from_numpy(scores).float(),
        labels=torch.from_numpy(labels).long(),
    )

mmengine.dump(results[0], 'ensemble.pkl')
log_stage('END wbf')


In [ ]:
%%writefile predict_mask.py

import mmcv
import mmengine
import torch
from mmengine.runner import load_checkpoint
from mmengine.structures.instance_data import InstanceData
from mmdet.registry import MODELS
from mmdet.structures import DetDataSample
from mmdet.structures.mask import encode_mask_results
from mmdet.utils import register_all_modules

register_all_modules()
cfg = mmengine.Config.fromfile('/kaggle/working/inputs/hubmap-2023-configs-patched/m0i.py')
model = MODELS.build(cfg.model)
load_checkpoint(model, '/kaggle/working/inputs/hubmap-2023-checkpoints/m1i.pth')
model.eval()
model.cuda()


@torch.no_grad()
def predict_mask(result, input_size=(1440, 1440)):
    img = mmcv.imread(result['img_path'])
    img = mmcv.imresize(img, input_size)
    batch_data = dict(
        inputs=torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).cuda(),
        data_samples=[
            DetDataSample(metainfo=dict(img_id=result['img_id'],
                                        ori_shape=(512, 512),
                                        img_shape=(1440, 1440),
                                        img_path=result['img_path'],
                                        scale_factor=(1440 / 512, 1440 / 512)))
        ])
    batch_data = model.data_preprocessor(batch_data, False)
    batch_data_inputs = batch_data['inputs']
    batch_data_samples = batch_data['data_samples']
    batch_img_metas = [
        data_samples.metainfo for data_samples in batch_data_samples
    ]
    img_feats = model.extract_feat(batch_data_inputs)

    img_result = InstanceData()
    for k, v in result['pred_instances'].items():
        img_result[k] = v.cuda()
    img_result.bboxes *= 1440 / 512
    results_list = model.roi_head.predict_mask(img_feats,
                                               batch_img_metas, [img_result],
                                               rescale=True)
    out = results_list[0].cpu()
    ret = dict(img_id=result['img_id'],
               ori_shape=(512, 512),
               img_shape=(1440, 1440),
               img_path=result['img_path'],
               scale_factor=(1440 / 512, 1440 / 512))
    ret['pred_instances'] = dict(
        bboxes=out['bboxes'],
        labels=out['labels'],
        scores=out['scores'],
        masks=encode_mask_results(out['masks'])
    )
    return ret


results = mmengine.load('/kaggle/working/ensemble.pkl')
outputs = []
for result in results:
    output = predict_mask(result)
    outputs.append(output)
mmengine.dump(outputs, '/kaggle/working/ensemble_results.pkl')

In [ ]:
log_stage('START predict_mask')
_proc = subprocess.run(
    ['python', 'predict_mask.py'],
    stderr=subprocess.PIPE,
)
with open('/kaggle/working/stderr_predict_mask.log', 'wb') as f:
    f.write(_proc.stderr or b'')
if _proc.returncode != 0:
    with open(ERR_PATH, 'a') as out:
        out.write('\n=== predict_mask.py stderr ===\n')
        out.write((_proc.stderr or b'').decode(errors='replace'))
    raise RuntimeError(f'predict_mask.py exit {_proc.returncode}')
log_stage('END predict_mask')


In [ ]:
import base64
import numpy as np
from pycocotools import _mask as coco_mask
import typing as t
import zlib


def encode_binary_mask(mask: np.ndarray) -> t.Text:
  """Converts a binary mask into OID challenge encoding ascii text."""

  # check input mask --
  if mask.dtype != bool:
    raise ValueError(
        "encode_binary_mask expects a binary mask, received dtype == %s" %
        mask.dtype)

  mask = np.squeeze(mask)
  if len(mask.shape) != 2:
    raise ValueError(
        "encode_binary_mask expects a 2d mask, received shape == %s" %
        mask.shape)

  # convert input mask to expected COCO API input --
  mask_to_encode = mask.reshape(mask.shape[0], mask.shape[1], 1)
  mask_to_encode = mask_to_encode.astype(np.uint8)
  mask_to_encode = np.asfortranarray(mask_to_encode)

  # RLE encode mask --
  encoded_mask = coco_mask.encode(mask_to_encode)[0]["counts"]

  # compress and base64 encoding --
  binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
  base64_str = base64.b64encode(binary_str)
  return base64_str


In [ ]:
log_stage('START encode_submission')
import os
import mmcv
import mmengine
import pandas as pd
import pycocotools.mask as mask_utils

results = mmengine.load('/kaggle/working/ensemble_results.pkl')
ids = []
HEIGHT = 512
WIDTH = 512
prediction_strings = []
for result in results:
    img_path = result['img_path']
    filename = os.path.basename(img_path)
    ids.append(filename[:-4])
    pred_instances = result['pred_instances']
    bboxes = pred_instances['bboxes']
    scores = pred_instances['scores'].tolist()
    labels = pred_instances['labels'].tolist()
    masks = pred_instances['masks']
    instance_strings = []
    for label, score, mask in zip(labels, scores, masks):
        if label != 0:
            continue
        mask = mask_utils.decode(mask).astype(bool)
        mask_string = encode_binary_mask(mask).decode('utf-8')
        
        instance_string = f'{label} {score} {mask_string}'
        instance_strings.append(instance_string)
    prediction_strings.append(' '.join(instance_strings))
log_stage('END encode_submission')


In [ ]:
sub = pd.DataFrame(dict(
    id=ids,
    height=[HEIGHT] * len(ids),
    width=[WIDTH] * len(ids),
    prediction_string=prediction_strings
))

In [ ]:
sub.to_csv('submission.csv', index=False)
import time as _t
print(f'Total pipeline elapsed: {(_t.time() - PIPELINE_T0) / 3600:.2f} hours')
log_stage('pipeline_done')
